# BLCKBAUX on a Colab T4

This notebook installs the prebuilt SM75 wheel. It does not compile Rust, CUDA, or PTX. Select a GPU runtime and run the cells in order. The release wheel targets Linux CPython 3.13.

In [ ]:
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), sys.version
WHEEL = (
    'https://github.com/Kvutza/blckbaux/releases/download/'
    'v0.2.3-cuda75-cp313/'
    'blckbaux-0.2.3+cuda75-cp313-cp313-manylinux_2_28_x86_64.whl'
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-deps', WHEEL],
    check=True,
)


In [ ]:
import jax
import torch
from blckbaux import capabilities

assert torch.cuda.is_available(), 'select a CUDA GPU runtime'
assert any(device.platform in {'gpu', 'cuda'} for device in jax.devices()), jax.devices()
cuda = next(item for item in capabilities(probe=True) if item.name == 'cuda')
assert cuda.available, cuda.reason
assert {'torch', 'jax'} <= set(cuda.array_frameworks), cuda.array_frameworks
print(cuda)


In [ ]:
import jax.numpy as jnp
import numpy as np
from blckbaux import ENN
from blckbaux.enn.enn_params import ENNParams
from blckbaux.turbo.config.surrogate import NeighborIndex

train_x = np.arange(48, dtype=float).reshape(16, 3) / 32
train_y = np.column_stack((np.sin(train_x[:, 0]), np.cos(train_x[:, 1])))
model = ENN(train_x, train_y, np.full_like(train_y, 0.01), index_driver=NeighborIndex.CUDA)
params = ENNParams(3, 0.7, 0.13)

torch_query = torch.tensor([[0.125, 0.25, 0.375]], device='cuda')
torch_snapshot = model.device_posterior(torch_query, params)
torch_mean = torch.from_dlpack(torch_snapshot.mean)
assert torch_mean.is_cuda

jax_device = next(device for device in jax.devices() if device.platform in {'gpu', 'cuda'})
jax_query = jax.device_put(jnp.asarray([[0.125, 0.25, 0.375]], dtype=jnp.float32), jax_device)
jax_snapshot = model.device_posterior(jax_query, params)
jax_mean = jax.dlpack.from_dlpack(jax_snapshot.mean, copy=False)
assert next(iter(jax_mean.devices())).platform in {'gpu', 'cuda'}
print('Torch and JAX CUDA resident posterior: ok')
